# Kütüphaneleri Yükleme

In [1]:
! pip install sweetviz
! pip install dice_ml



# Veri Görselleştirme

In [1]:
import sweetviz as sv
import pandas as pd

# Veri setini yükleyin
df = pd.read_csv("diabetes.csv")

# Sweetviz raporu oluşturun
report = sv.analyze(df)

# Raporu HTML formatında kaydedin ve görüntüleyin
report.show_html("diabetes_sweetviz_report.html")


                                             |          | [  0%]   00:00 -> (? left)

Report diabetes_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


0:131: execution error: bazı nesneler dosyası bulunamadı. (-43)


In [3]:
print(df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']].describe())


          Glucose  BloodPressure  SkinThickness     Insulin         BMI
count  768.000000     768.000000     768.000000  768.000000  768.000000
mean   120.894531      69.105469      20.536458   79.799479   31.992578
std     31.972618      19.355807      15.952218  115.244002    7.884160
min      0.000000       0.000000       0.000000    0.000000    0.000000
25%     99.000000      62.000000       0.000000    0.000000   27.300000
50%    117.000000      72.000000      23.000000   30.500000   32.000000
75%    140.250000      80.000000      32.000000  127.250000   36.600000
max    199.000000     122.000000      99.000000  846.000000   67.100000


# xGBoost Model Geliştirme

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

# 1. Veri setini yükleyin
df = pd.read_csv("diabetes.csv")

# 2. Özellikleri (X) ve hedef değişkeni (y) ayırın
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

# Veri türlerini kontrol edin ve uygun hale getirin
for col in X.columns:
    if X[col].dtype == "object":
        try:
            X[col] = X[col].astype(float)  # Sayısal değerlere dönüştürme
        except ValueError:
            print(f"{col} sütunu sayısal değerlere dönüştürülemedi.")

# 3. Eğitim ve test veri setlerine bölme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Ölçeklendirme işlemi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Optimize edilmiş XGBoost modeli oluşturun
model = XGBClassifier(
    max_depth=4,              # Daha sığ ağaçlar
    learning_rate=0.001,      # Daha düşük öğrenme oranı
    n_estimators=100000,      # Daha fazla ağaç sayısı
    subsample=0.6,            # Alt örnekleme oranı (daha düşük)
    colsample_bytree=0.6,     # Özelliklerin alt örnekleme oranı (daha düşük)
    min_child_weight=10,      # Bölünme için minimum ağırlık
    gamma=5,                  # Bölünme için minimum kazanç artırıldı
    reg_lambda=10,            # L2 düzenleme (ridge regression)
    reg_alpha=2,              # L1 düzenleme (lasso regression)
    random_state=42,          # Rastgelelik kontrolü
    eval_metric="logloss",  # Değerlendirme metriği
    use_label_encoder=False   # Etiket kodlama uyumluluğunu kapatma
)

# 6. Modeli eğitin
model.fit(X_train_scaled, y_train)

# 7. Tahminler yapın
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# 8. Performans metriklerini hesaplayın ve yazdırın
print("=== Eğitim Verisi Performans ===")
print(f"Doğruluk (Accuracy): {accuracy_score(y_train, y_train_pred):.2f}")
print(classification_report(y_train, y_train_pred))
print("Karmaşıklık Matrisi (Confusion Matrix):")
print(confusion_matrix(y_train, y_train_pred))

print("\n=== Test Verisi Performans ===")
print(f"Doğruluk (Accuracy): {accuracy_score(y_test, y_test_pred):.2f}")
print(classification_report(y_test, y_test_pred))
print("Karmaşıklık Matrisi (Confusion Matrix):")
print(confusion_matrix(y_test, y_test_pred))


=== Eğitim Verisi Performans ===
Doğruluk (Accuracy): 0.79
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       400
           1       0.76      0.57      0.65       214

    accuracy                           0.79       614
   macro avg       0.78      0.74      0.75       614
weighted avg       0.78      0.79      0.78       614

Karmaşıklık Matrisi (Confusion Matrix):
[[361  39]
 [ 91 123]]

=== Test Verisi Performans ===
Doğruluk (Accuracy): 0.76
              precision    recall  f1-score   support

           0       0.79      0.86      0.82       100
           1       0.69      0.57      0.63        54

    accuracy                           0.76       154
   macro avg       0.74      0.72      0.72       154
weighted avg       0.75      0.76      0.75       154

Karmaşıklık Matrisi (Confusion Matrix):
[[86 14]
 [23 31]]


# True Positive Gözlem Seçme

In [5]:
# 9. True positive örnekleri seç
true_positives = X_test[(y_test == 1) & (y_test_pred == 1)]

# True positive örneklerini yazdır
print("=== True Positive Örnekleri ===")
print(true_positives.head(20))
true_positive_indices = true_positives.index[[0, 3, 9]]  # 0, 3 ve 9. gözlemi seçmek
actionable_features = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Seçilen True Positive Indices:", true_positive_indices)

=== True Positive Örnekleri ===
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
408            8      197             74              0        0  25.9   
13             1      189             60             23      846  30.1   
245            9      184             85             15        0  30.0   
480            3      158             70             30      328  35.5   
746            1      147             94             41        0  49.3   
415            3      173             84             33      474  35.7   
484            0      145              0              0        0  44.2   
712           10      129             62             36        0  41.2   
355            9      165             88              0        0  30.4   
440            0      189            104             25        0  34.3   
235            4      171             72              0        0  43.6   
332            1      180              0              0        0  43.3   
743   

# Counterfactual Explanation

In [6]:
import dice_ml

# Dice için veri hazırlama
data_combined = pd.concat([
    pd.DataFrame(X_train_scaled, columns=X.columns).assign(Outcome=y_train.values),
    pd.DataFrame(X_test_scaled, columns=X.columns).assign(Outcome=y_test.values)
])

data_dice = dice_ml.Data(
    dataframe=data_combined,
    continuous_features=X.columns.tolist(),
    outcome_name="Outcome"
)

# Modeli Dice için sarmalama
model_dice = dice_ml.Model(model=model, backend="sklearn")

# "method" parametresine sadece 'genetic' diyerek explainer oluşturun
explainer = dice_ml.Dice(
    data_interface=data_dice,
    model_interface=model_dice,
    method="genetic"
)

# Counterfactual açıklama üretimi için veri
input_datapoint = pd.DataFrame(
    scaler.transform(df.loc[[24, 440, 480]].drop(columns=["Outcome"])), 
    columns=X.columns
)

# permitted_range_scaled söz konusu feature’lar için min/max değerleri içeriyor
permitted_range_scaled = {
    "Glucose": scaler.transform([[0, 70, 0, 0, 0, 0, 0, 0], [0, 180, 0, 0, 0, 0, 0, 0]])[:, 1].tolist(),
    "BloodPressure": scaler.transform([[0, 0, 60, 0, 0, 0, 0, 0], [0, 0, 120, 0, 0, 0, 0, 0]])[:, 2].tolist(),
    "SkinThickness": scaler.transform([[0, 0, 0, 10, 0, 0, 0, 0], [0, 0, 0, 50, 0, 0, 0, 0]])[:, 3].tolist(),
    "Insulin": scaler.transform([[0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 200, 0, 0, 0]])[:, 4].tolist(),
    "BMI": scaler.transform([[0, 0, 0, 0, 0, 18.5, 0, 0], [0, 0, 0, 0, 0, 50, 0, 0]])[:, 5].tolist()
}

# generate_counterfactuals içerisinde genetik algoritma parametrelerini giriyoruz:
cf_conditional = explainer.generate_counterfactuals(
    query_instances=input_datapoint,
    total_CFs=10,
    desired_class="opposite",
    permitted_range=permitted_range_scaled,
    features_to_vary=["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
)



In [7]:
from IPython.display import display
import pandas as pd
import numpy as np

def format_instance(df):
    """Bir DataFrame'deki sayısal değerleri formatlar ve eksik olanları '-' ile doldurur."""
    formatted_df = df.copy()
    for col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].apply(lambda x: f"{x:.2f}" if pd.notnull(x) else "-")
    return formatted_df

def mark_unchanged_columns(cf_df, original_df):
    """
    Counterfactual DataFrame'deki orijinal ile aynı olan değerleri '-' ile işaretler.
    Orijinal ve Counterfactual DataFrame sayısal olarak karşılaştırılır.
    """
    marked_df = cf_df.copy()
    for col in cf_df.columns:
        original_value = original_df.iloc[0][col]  # İlk satırdaki orijinal değer
        marked_df[col] = cf_df[col].apply(
            lambda x: f"{x:.2f}" if pd.notnull(x) and not abs(x - original_value) < 1e-2 else "-"
        )
    return marked_df

# Bütün counterfactual örneklerini listeye dönüştürme ve inverse scaling
scaled_results = []

for cf_example in cf_conditional.cf_examples_list:
    # Orijinal gözlemi inverse scale işlemi ile al
    original_instance = cf_example.test_instance_df
    numeric_columns = [col for col in X.columns if col in original_instance.columns]
    original_numeric = original_instance[numeric_columns].astype(float)
    inverse_scaled_original = scaler.inverse_transform(original_numeric)
    inverse_scaled_original_df = pd.DataFrame(inverse_scaled_original, columns=numeric_columns)
    
    # Counterfactual'ları inverse scale işlemiyle al
    cf_df = pd.DataFrame(cf_example.final_cfs_df)
    cf_numeric = cf_df[numeric_columns].replace('-', np.nan).fillna(0).astype(float)
    inverse_scaled_cf = scaler.inverse_transform(cf_numeric)
    inverse_scaled_df = pd.DataFrame(inverse_scaled_cf, columns=numeric_columns)
    
    # Sonuçları listeye ekle
    scaled_results.append((inverse_scaled_df, inverse_scaled_original_df))

# scaled_results listesindeki her bir sonucu düzgün formatta yazdırma
for i, (scaled_df, scaled_original) in enumerate(scaled_results):
    # Orijinal gözlemi ve counterfactual seti formatla
    formatted_original = format_instance(scaled_original)
    marked_cf = mark_unchanged_columns(scaled_df, scaled_original)

    # Orijinal gözlemi ve counterfactual seti yazdır
    print(f"\nQuery instance (original outcome: {i}):")
    display(formatted_original.style.set_table_styles(
        [{'selector': 'th', 'props': [('border', '1px solid black')]},
         {'selector': 'td', 'props': [('border', '1px solid black')]}]
    ).hide_index())

    print(f"\nDiverse Counterfactual set (new outcome: 0):")
    display(marked_cf.style.set_table_styles(
        [{'selector': 'th', 'props': [('border', '1px solid black')]},
         {'selector': 'td', 'props': [('border', '1px solid black')]}]
    ).hide_index())



Query instance (original outcome: 0):


/var/folders/01/_bgw0fxs5rv8ysw_gs46q0v40000gn/T/ipykernel_13100/3551539517.py:56: FutureWarning: this method is deprecated in favour of `Styler.hide(axis="index")`
  ).hide_index())


Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
11.00,143.00,94.00,33.00,146.00,36.60,0.25,51.00



Diverse Counterfactual set (new outcome: 0):


/var/folders/01/_bgw0fxs5rv8ysw_gs46q0v40000gn/T/ipykernel_13100/3551539517.py:62: FutureWarning: this method is deprecated in favour of `Styler.hide(axis="index")`
  ).hide_index())


...

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
-,122.00,94.00,-,5.00,32.80,-,-
-,110.00,92.00,-,21.14,37.60,-,-
-,110.00,85.00,-,5.00,37.40,-,-
-,144.00,-,28.00,140.00,29.50,-,-
-,115.00,86.00,-,21.14,38.40,-,-
-,117.00,92.00,-,81.44,34.10,-,-
-,111.00,86.00,-,5.00,27.00,-,-
-,115.00,92.00,-,96.00,34.60,-,-
-,124.00,74.00,-,32.61,34.00,-,-
-,141.00,74.00,-,35.08,27.60,-,-



Query instance (original outcome: 2):


Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
3.00,158.00,70.00,30.00,328.00,35.50,0.34,35.00



Diverse Counterfactual set (new outcome: 0):


Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
-,112.00,-,-,158.00,39.40,-,-
-,106.00,-,-,158.00,39.40,-,-
-,120.00,-,-,135.00,42.90,-,-
-,115.00,-,-,96.00,34.60,-,-
-,115.00,-,-,96.00,27.60,-,-
-,115.00,-,-,5.00,34.80,-,-
-,70.00,-,-,5.00,35.00,-,-
-,70.00,-,-,5.00,18.50,-,-
-,146.00,-,38.00,167.97,28.00,-,-
-,112.00,72.00,-,176.00,34.40,-,-
